# Fase 4 v2 — 01: baseline (camino C) overlay cenital vs lateral

Evidencia **visual** del baseline: proyecta el template del campo sobre cada frame
vía `inv(H)` y dibuja líneas + landmarks. Confirma cualitativamente lo que el nb00
midió: camino C alinea en cenital y se rompe/ausenta en lateral.

Salida: overlays + `outputs/baseline_overlays.zip` (descargable).

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import numpy as np
import cv2, decord

# repo raiz: sube desde el cwd hasta encontrar src/core (portable a cualquier clon)
REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core import field_template as ft
from src.core import field_landmarks as fl
from src.core import homography_metrics as hm
from src.core import auto_homography as ah

OUT = REPO / 'notebooks/fase_4_homografia/v2_robust/outputs'
OVL = OUT / 'baseline_overlays'
if OVL.exists(): shutil.rmtree(OVL)
OVL.mkdir(parents=True, exist_ok=True)
EVAL = json.load(open(OUT / 'eval_clips.json'))
print('REPO:', REPO)
print('clips:', list(EVAL.keys()))

In [ ]:
# --- baseline solver (igual que nb00) ---
INNER = ['inner_tl','inner_tr','inner_br','inner_bl']
INNER_CM = np.array([fl.LANDMARK_POINTS[n] for n in INNER], dtype=np.float64)

def baseline_H(frame_rgb):
    bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    try: res = ah.solve(bgr)
    except Exception: return None
    H = getattr(res,'H',None)
    if H is None or not getattr(res,'ok',False): return None
    return np.asarray(H, dtype=np.float64)

# --- overlay: cm -> imagen via inv(H) ---
def cm_to_img(Hinv, pts_cm):
    p = np.asarray(pts_cm, np.float64).reshape(-1,1,2)
    return cv2.perspectiveTransform(p, Hinv).reshape(-1,2)

def draw_overlay(frame_rgb, H):
    img = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).copy()
    if H is None:
        cv2.putText(img,'NO H',(40,80),cv2.FONT_HERSHEY_SIMPLEX,2.0,(0,0,255),4,cv2.LINE_AA)
        return img
    Hinv = np.linalg.inv(H)
    L = fl  # alias
    # rectangulo interior
    rect = [L.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    poly = cm_to_img(Hinv, rect).astype(np.int32)
    cv2.polylines(img,[poly],True,(0,255,0),2,cv2.LINE_AA)
    # linea central
    cl = cm_to_img(Hinv,[L.LANDMARK_POINTS['center_top'],L.LANDMARK_POINTS['center_bot']]).astype(np.int32)
    cv2.line(img,tuple(cl[0]),tuple(cl[1]),(0,255,255),2,cv2.LINE_AA)
    # circulo central (muestreado)
    cx,cy,r = L.CENTER_CIRCLE
    th = np.linspace(0,2*np.pi,48)
    circ = np.stack([cx+r*np.cos(th), cy+r*np.sin(th)],1)
    cpx = cm_to_img(Hinv,circ).astype(np.int32)
    cv2.polylines(img,[cpx],True,(255,128,0),2,cv2.LINE_AA)
    # landmarks
    for name,pt in L.LANDMARK_POINTS.items():
        q = cm_to_img(Hinv,[pt])[0]
        if np.all(np.isfinite(q)):
            cv2.circle(img,(int(q[0]),int(q[1])),4,(255,0,255),-1,cv2.LINE_AA)
    return img

In [ ]:
# 3 frames por clip (inicio/medio/fin), overlay, guardar
saved = 0
for name, m in EVAL.items():
    path = Path({'cenital_9933':'/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV',
                 'cenital_9938':'/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9938.MOV',
                 'lateral_9913':'/workspace/Meta_Glasses/18abril/Cámaras/IMG_9913.MOV',
                 'lateral_9914':'/workspace/Meta_Glasses/18abril/Cámaras/IMG_9914.MOV',
                 'lateral_9915':'/workspace/Meta_Glasses/18abril/Cámaras/IMG_9915.MOV'}[name])
    vr = decord.VideoReader(str(path)); n = len(vr)
    for tag, idx in [('a',int(n*0.1)),('b',int(n*0.5)),('c',int(n*0.9))]:
        fr = vr[idx].asnumpy()
        ov = draw_overlay(fr, baseline_H(fr))
        # reducir para zip liviano
        h,w = ov.shape[:2]; nw=900; ov=cv2.resize(ov,(nw,int(h*nw/w)))
        cv2.imwrite(str(OVL/f'{name}_{tag}.jpg'), ov, [cv2.IMWRITE_JPEG_QUALITY,85]); saved+=1
    print('ok', name)
print('overlays guardados:', saved)
zb = str(OUT/'baseline_overlays')
if os.path.exists(zb+'.zip'): os.remove(zb+'.zip')
shutil.make_archive(zb,'zip',OVL)
print('ZIP:', zb+'.zip', round(os.path.getsize(zb+'.zip')/1e6,2),'MB')

## Resumen

Overlays del baseline (camino C). Esperado: cenital alineado, lateral roto/ausente.
Esto fija la vara que el solver multi-feature (nb 02) debe superar.